# 1/2 · Build `dist_3G` on Google Colab (OpenCellID + Collins Bartholomew)

This is a **Python notebook that runs R** via the `rpy2` `%%R` magic. It runs
**`Cleaning OCI and CB data - district level.R`** → writes `Clean data/dist_3G.Rda`
(district-level 3G coverage on the 2011 LFS frame) to your Drive.

**Run this notebook FIRST** — the MCL notebook (2/2) needs the `dist_3G.Rda` it produces.

### Before you start
1. Copy the `…/3G-FLFP/Code` folder to Google Drive (`Raw Data/`, `Clean data/`, the `.R` scripts).
2. **Runtime → Change runtime type → High-RAM** (no GPU — nothing here uses one). The coverage step is memory-heavy.
3. Run top to bottom. Step 4 takes ~30–60 min; a spinner with no output is normal (`%%R` prints only when finished).

## 1. Install the R geospatial stack

Colab ships R + `rpy2`. Add the system libs, then install the R packages as pre-built Linux binaries (fast).

In [ ]:
# system libraries the R geospatial packages compile / link against (~1-2 min)
!apt-get update -qq
!apt-get install -y -qq gdal-bin libgdal-dev libgeos-dev libproj-dev libudunits2-dev libssl-dev libxml2-dev
!ldconfig
!ldconfig -p | grep -E "libproj|libgdal|libgeos" | head

In [ ]:
# load the R cell magic -> %%R runs R in a persistent embedded R session
%load_ext rpy2.ipython

In [ ]:
%%R
# Colab's R has no matching pre-built binaries on the CRAN mirrors, so the
# geospatial packages (sf / terra / exactextractr) are compiled FROM SOURCE
# against the system GDAL/PROJ/GEOS installed in the apt cell above.
# First run ~10-15 min (mostly the geospatial trio); nothing to do on re-runs.
options(repos = c(CRAN = "https://cloud.r-project.org"))
pkgs <- c("tidyverse","data.table","stringi","haven","readxl","lubridate",
          "sf","terra","exactextractr")
need <- setdiff(pkgs, rownames(installed.packages()))
if (length(need)) install.packages(need)
# the geospatial trio must load against this runtime's system libs; if a
# mismatched copy is present (e.g. from an earlier attempt), remove it and
# recompile from source -- remove.packages is the step that guarantees the
# broken .so is actually replaced.
geo <- c("sf","terra","exactextractr")
ok <- tryCatch({ suppressMessages(lapply(geo, library, character.only = TRUE)); TRUE },
               error = function(e) { message("geo load failed: ", conditionMessage(e)); FALSE })
if (!ok) {
  message(">> recompiling sf / terra / exactextractr from source")
  suppressWarnings(remove.packages(intersect(geo, rownames(installed.packages()))))
  install.packages(geo)
  suppressMessages(lapply(geo, library, character.only = TRUE))
}
suppressMessages(lapply(setdiff(pkgs, geo), library, character.only = TRUE))
cat("packages ready:\n")
invisible(lapply(pkgs, function(p) cat(sprintf("  %-14s %s\n", p, requireNamespace(p, quietly = TRUE)))))

## 2. Mount Google Drive and point at the project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# >>> EDIT to the Code folder of your project on Drive <<<
PROJECT_DIR = "/content/drive/MyDrive/3G - FLFP/3G-FLFP/Code"

assert os.path.isdir(PROJECT_DIR), f"Not found: {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
%%R -i PROJECT_DIR
setwd(PROJECT_DIR)

# Linux is case-sensitive; the scripts mix "Raw Data"/"Raw data" and "Clean data".
# Create alias symlinks so either casing resolves, and make sure Clean data exists.
alias <- function(real, link) if (dir.exists(real) && !file.exists(link)) file.symlink(normalizePath(real), link)
alias("Raw Data", "Raw data"); alias("Raw data", "Raw Data")
alias("Clean data", "Clean Data"); alias("Clean Data", "Clean data")
if (!dir.exists("Clean data")) dir.create("Clean data")

need <- c("Cleaning OCI and CB data - district level.R",
          "Raw Data/Vietnam_Cell_tower.csv",
          "Raw Data/VNShapefile/gadm41_VNM_shp/gadm41_VNM_2.shp",
          "Raw Data/rural_urban_wards.csv",
          "Raw Data/LFS/lfs_dist_11.csv",
          "Raw Data/Collins Bartholomew/wx910xj1289/MCE_Global3G_2012.tif",
          "Raw Data/Population Data/vnm_ppp_2017.tif")
cat("input check (MISSING = fix before running):\n")
for (f in need) cat(sprintf("  [%-7s] %s\n", ifelse(file.exists(f), "OK", "MISSING"), f))

## 3. Helper — run a project `.R` file with Colab-friendly paths

In [ ]:
%%R
# Source a project .R file but redirect its hardcoded Windows Overleaf figure
# folder to a local ./Figures dir, and drop any Windows setwd().
run_r_script <- function(path) {
  stopifnot(file.exists(path))
  dir.create("Figures", showWarnings = FALSE)
  code <- readLines(path, encoding = "UTF-8", warn = FALSE)
  code <- gsub("C:/Users/Anri Sakakibara/Dropbox/Apps/Overleaf/3G in Vietnam/Figures/Descriptive Stats",
               "Figures", code, fixed = TRUE)
  code <- code[!grepl("^\\s*setwd\\s*\\(", code)]
  message("Running: ", path)
  t0 <- Sys.time()
  eval(parse(text = paste(code, collapse = "\n")), envir = globalenv())
  message("Finished ", path, " in ",
          round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 1), " min")
}
cat("run_r_script() ready\n")

## 4. Build `dist_3G` — the long step (~30–60 min)

Writes `Clean data/dist_3G.Rda` (+ `.dta`), `oci_dist_1017.*`, `cb_dist_1017.*` to your Drive.

In [ ]:
%%R
run_r_script("Cleaning OCI and CB data - district level.R")

## 5. Check the outputs

Saved to your Drive (persist after the runtime ends). These feed the MCL notebook.

In [ ]:
import os
for f in ["Clean data/dist_3G.Rda", "Clean data/dist_3G.dta", "Clean data/oci_dist_1017.Rda", "Clean data/cb_dist_1017.Rda"]:
    ok = os.path.exists(f)
    size = f" ({os.path.getsize(f)/1e6:.1f} MB)" if ok else ""
    print(("OK  " if ok else "--- ") + f + size)